<a id="top"></a>
# Ocean Satellite Data for Florida: Your First Colab Notebook

**Tutorial for undergraduate STEM students. No coding experience needed.**

## Summary

Warm water stresses corals, and murky or green water can mean runoff and algae. Satellites measure both from space, every day. In this notebook you will make **two maps of Florida**:

1. **Sea surface temperature (SST)** from **GHRSST MUR**, a NASA product that blends many satellites into one 1 km map
2. **Chlorophyll-a** from **PACE**, NASA's newest ocean-color satellite (launched 2024). Chlorophyll is a measure of how much microscopic algae (phytoplankton) is in the water.

You will use a Python package called `earthaccess` to find NASA data and open it directly from the cloud, **with no downloading**, the same way scientists do.

This notebook has **two parts**. Part 0 is a short Colab tutorial. Then you build the maps, step by step.

## Learning Objectives

By the end of this notebook you will know:

- How to use Google Colab: text cells, code cells, running, and saving
- How to log in to NASA Earthdata with `earthaccess`
- How to list NASA datasets and search for satellite files with `earthaccess`
- How to open data from the cloud without downloading it
- How to make a map of SST and a map of chlorophyll

## Contents

**Part 0: Colab Crash Course**
- [0.1 What is Colab?](#0.1-What-is-Colab?)
- [0.2 Text cells and code cells](#0.2-Text-cells-and-code-cells)
- [0.3 Run your first code](#0.3-Run-your-first-code)
- [0.4 Things that trip up beginners](#0.4-Things-that-trip-up-beginners)

**Part 1: Making the Maps**
- [1. Setup](#1.-Setup)
- [2. NASA Earthdata login](#2.-NASA-Earthdata-login)
- [3. What data is available?](#3.-What-data-is-available?)
- [4. Choose a place and a time](#4.-Choose-a-place-and-a-time)
- [5. Chlorophyll map (PACE)](#5.-Chlorophyll-map-(PACE))
- [6. Sea surface temperature map (GHRSST MUR)](#6.-Sea-surface-temperature-map-(GHRSST-MUR))
- [7. Save your maps](#7.-Save-your-maps)
- [8. If something goes wrong](#8.-If-something-goes-wrong)

**Before you start:** make a free [NASA Earthdata Login](https://urs.earthdata.nasa.gov/) account (click *Register*). You will need the username and password in Step 2.

---
# Part 0: Colab Crash Course

## 0.1 What is Colab?

**Google Colab** is a free notebook that runs Python **in your web browser**. You don't install anything. Google gives you a temporary computer in the cloud, and your browser is the remote control.

A **notebook** is a document that mixes **text** (like this) with **code** you can run, and shows the results (numbers, plots, maps) right below the code.

What you need: a Google account, and a browser (Chrome works best).

> 🔑 **The one big idea:** the computer in the cloud is **temporary**. When you close the tab and come back later, it is erased: your downloaded files and everything you ran are gone. Your **notebook** is saved to your Google Drive, but you have to **run the code again** each time. Download any results you want to keep (we do this in Step 7).

[back to top](#top)

## 0.2 Text cells and code cells

A notebook is made of **cells**. There are two kinds:

| | Text cell | Code cell |
|---|---|---|
| Looks like | Plain writing (like this one) | A gray box with a ▶ play button on the left |
| What's in it | Explanations | Python code |
| How to use it | Just read it. Double-click to edit. | **Run** it |

**How to run a code cell** (any of these):
- Click the **▶ play button** on the left of the cell
- Or click inside the cell and press **Shift + Enter** (this runs the cell *and* jumps to the next one, which is the fastest way to work through a notebook)

While it runs, you'll see a spinning circle. When it's done, you'll see a check mark ✓ and the output below the cell.

**Run the cells in order, from top to bottom.** Each cell builds on the ones before it. If you skip one, later cells will show errors.

[back to top](#top)

## 0.3 Run your first code

Try it now. Click the cell below and press **Shift + Enter**. The first time, Colab takes a few seconds to connect to its cloud computer. (It may show *Connecting...* in the top right.)

In [ ]:
print("Hello, ocean!")

You should see `Hello, ocean!` appear below the cell. 🎉 You just ran Python.

Now try some math. This cell **stores** numbers in named boxes (called *variables*) and then uses them. Run it:

In [ ]:
temp_celsius = 29.5                              # a hot summer day on the water
temp_fahrenheit = temp_celsius * 9 / 5 + 32      # the conversion formula

print("Water temperature:", temp_celsius, "°C")
print("In Fahrenheit:", temp_fahrenheit, "°F")

> ✏️ **Your turn:** change `29.5` to `31.2`, then run the cell again. The Fahrenheit number updates. Lines starting with `#` are **comments**: notes for humans that Python ignores.

[back to top](#top)

## 0.4 Things that trip up beginners

**Red text under a cell = an error.** Don't panic. Read the **last line** of the error: it usually says what's wrong. The most common causes are:
- You skipped an earlier cell (go back and run the cells above, in order)
- A typo, such as a missing quote `"` or a capital letter in the wrong place
- The Colab session was reset (see below)

**"Restart session" or "Runtime disconnected".** Colab disconnects if you leave it idle for a while. When you come back, go to the menu **Runtime → Run all** to re-run everything from the top.

**Useful menu items** (under **Runtime** at the top):
| Menu item | What it does |
|---|---|
| **Run all** | Runs every cell, top to bottom |
| **Restart session** | Wipes the memory and starts fresh. Use it when things get weird. |
| **Disconnect and delete runtime** | Throws away the cloud computer when you're done |

**The Files panel.** Click the 📁 **folder icon** on the left edge. This shows the files on your temporary cloud computer. Maps and data you create will show up here. Right-click a file to download it.

**Lines that start with `!`** run a command on the cloud computer instead of Python. We use `!pip install` to add a package.

**Saving.** Colab saves the notebook itself automatically to your Drive. To keep your own copy of this notebook, use **File → Save a copy in Drive**.

You are ready. On to the real thing.

[back to top](#top)

---
# Part 1: Making the Maps

## 1. Setup

First we install the packages we need. `earthaccess` talks to NASA, and `cartopy` draws maps with coastlines. This takes about a minute, and you'll see a lot of text scroll by, which is normal.

> ⚠️ If Colab says **"You must restart the runtime in order to use the newly installed versions"**, click **Restart session**, then continue from the next cell (you don't need to run the install cell again).

In [ ]:
!pip install -q -U earthaccess cartopy h5netcdf

Next we **import** the packages. This loads their tools into our notebook, and we give them short nicknames (`np`, `xr`, `plt`) so we type less.

| Package | What we use it for |
|---|---|
| `earthaccess` | Log in to NASA, search for data, open it |
| `xarray` (`xr`) | Work with satellite data (grids of numbers) |
| `numpy` (`np`) | Math on grids of numbers |
| `matplotlib` (`plt`) | Make plots |
| `cartopy` (`ccrs`) | Draw maps with coastlines |

In [ ]:
import os
import warnings

import earthaccess
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import LogNorm

warnings.filterwarnings("ignore", category=FutureWarning)   # hide harmless "future version" notices

print("Packages loaded ✓")

[back to top](#top)

## 2. NASA Earthdata login

NASA's satellite data is free, but you need to log in so NASA knows who is using it. When you run the next cell, a box appears asking for your **Earthdata username** and then your **password** (it shows as dots while you type). Type them in and press Enter.

In [ ]:
auth = earthaccess.login(persist=True)
print("Logged in?", auth.authenticated)

You should see `Logged in? True`. If it says `False`, check your username and password and run the cell again.

[back to top](#top)

## 3. What data is available?

NASA has hundreds of datasets. Each one has a **short name**, like an ID. We can ask `earthaccess` to list the datasets from one instrument. **OCI** is the ocean color camera on the PACE satellite.

In [ ]:
results = earthaccess.search_datasets(instrument="oci")
for item in results:
    summary = item.summary
    print(summary["short-name"])

That's a long list! Names are built from pieces. Look at `PACE_OCI_L3M_BGC`:

| Piece | Meaning |
|---|---|
| `PACE_OCI` | The satellite and camera |
| `L3M` | **Level 3 Mapped**: already placed on a regular latitude/longitude grid, so it's easy to map |
| `BGC` | **Biogeochemistry**: a bundle of ocean-life variables, including chlorophyll |

For sea surface temperature we'll use a dataset from a different NASA center, called **GHRSST MUR**. We can check its short name the same way:

In [ ]:
results = earthaccess.search_datasets(short_name="MUR-JPL-L4-GLOB-v4.1")
for item in results:
    summary = item.summary
    print(summary["short-name"])

So our two datasets are:

| Short name | What it is |
|---|---|
| `PACE_OCI_L3M_BGC` | PACE **biogeochemistry** bundle, which includes **chlorophyll** (how much algae is in the water) |
| `MUR-JPL-L4-GLOB-v4.1` | GHRSST MUR **sea surface temperature**, a complete map of the world every day at 1 km |

[back to top](#top)

## 4. Choose a place and a time

Everything you can change is in this one cell. We pick:
- a **box** around Florida (latitude and longitude limits; longitudes west of Greenwich are **negative**)
- a **time span**, `tspan`: a start date and an end date. We chose one week in early September 2026, near the warmest time of year for Florida's waters.

> ✏️ **Try later:** change the box to just the Florida Keys, `lat = (24.2, 25.9)` and `lon = (-83.2, -79.9)`, and re-run from here.

In [ ]:
# Latitude (south, north) and longitude (west, east) in degrees
lat = (24.0, 31.0)
lon = (-88.0, -79.8)

# Time span: ("start", "end") as "YYYY-MM-DD"
tspan = ("2026-09-01", "2026-09-07")

os.makedirs("maps", exist_ok=True)   # a folder for the pictures we make

[back to top](#top)

## 5. Chlorophyll map (PACE)

### 5.1 Search for the data

Now we search for **files**, not datasets. Each file is called a **granule**. We give `search_data` the short name, the time span, and a `granule_name` filter. The `*` is a wildcard that matches anything, so `"*.8D.*.4km.*"` means "8-day composite files at 4 km resolution".

Why 8 days? Satellites can't see through clouds, so NASA combines 8 days of images into one map to fill in the gaps.

In [ ]:
results = earthaccess.search_data(
    short_name="PACE_OCI_L3M_BGC",
    temporal=tspan,
    granule_name="*.8D.*.4km.*",
)
print("Files found:", len(results))

for granule in results:
    print(granule.data_links()[0].split("/")[-1])      # just the file name

Look at a file name, such as `PACE_OCI.20260829_20260905.L3m.8D.BGC.V3_2.4km.nc`. You can read it like a label:

| Piece | Meaning |
|---|---|
| `20260829_20260905` | The 8 days it covers (start and end date) |
| `L3m` | Level 3, mapped |
| `8D` | 8-day composite |
| `BGC` | Biogeochemistry bundle |
| `4km` | Pixel size |

Your time span may touch two 8-day files. That's fine: we'll use the first one.

> If you see `Files found: 0`, check the dates in Step 4: PACE data start in March 2024, and the newest days may not be processed yet. Try a week from a few weeks ago.

### 5.2 Open the data (no download needed!)

Here's the neat part. We **don't download** the file. `earthaccess.open` connects to it in NASA's cloud, and Python only pulls in the numbers we actually ask for.

In [ ]:
paths = earthaccess.open(results[:1])      # [:1] means "just the first file"

### 5.3 Cut out Florida

We open the file with `xarray`. The BGC bundle holds several variables. Let's look at the list:

In [ ]:
dataset = xr.open_dataset(paths[0])
print(list(dataset.data_vars))

The one we want is **`chlor_a`** (chlorophyll-a). Now we cut out our Florida box with `.sel`, and `.load()` reads just those numbers.

> 🧭 **A gotcha:** in PACE files, latitude is listed from **north to south**. That's why the latitude slice below is written `lat[1], lat[0]` (north first).

In [ ]:
chl = dataset["chlor_a"]
chl = chl.sel(lat=slice(lat[1], lat[0]), lon=slice(lon[0], lon[1]))   # cut out Florida
chl = chl.load()                                                        # read the numbers

print("Florida box:", dict(chl.sizes))
print(f"Pixels with data: {100 * float(np.isfinite(chl).mean()):.0f}%  (the rest are land or clouds)")

### 5.4 Make the map 🗺️

Chlorophyll values span a huge range, from about 0.05 in clear blue water to 10+ near the coast. So we use a **logarithmic color scale** (`LogNorm`), where each step in color is a *multiplication* instead of an addition. That lets us see detail in both clear and green water.

In [ ]:
fig = plt.figure(figsize=(8, 7))
ax = plt.axes(projection=ccrs.PlateCarree())

mesh = ax.pcolormesh(chl.lon, chl.lat, chl, cmap="viridis",
                     norm=LogNorm(vmin=0.05, vmax=10),
                     transform=ccrs.PlateCarree())

ax.add_feature(cfeature.LAND, facecolor="#e6dcc8", zorder=2)
ax.coastlines(resolution="10m", linewidth=0.6, zorder=3)
gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
gl.top_labels = gl.right_labels = False

ax.set_extent([lon[0], lon[1], lat[0], lat[1]], crs=ccrs.PlateCarree())
plt.colorbar(mesh, ax=ax, shrink=0.75, extend="both", label="Chlorophyll-a (mg/m³)")
ax.set_title(f"Chlorophyll-a, Florida, 8-day composite near {tspan[0]}\nPACE OCI (4 km)")

fig.savefig("maps/florida_chlorophyll.png", dpi=150, bbox_inches="tight")
plt.show()

**What do you see?**
- Where is the water greenest (high chlorophyll)? Near the coast or far from it?
- White areas are **gaps** (land, or clouds for all 8 days). Why might there be more gaps near the coast?

> ⚠️ **Careful near shore:** in shallow, clear water, the satellite can see sand and seagrass on the bottom and mistake that for chlorophyll. Be skeptical of very bright values hugging the coastline.

[back to top](#top)

## 6. Sea surface temperature map (GHRSST MUR)

Same recipe, new dataset: **search → open → cut out Florida → plot**.

### 6.1 Search and open

The MUR files are daily, so the same `tspan` gives about 7 files (NASA's search sometimes includes a day at the edge, which is fine). We search and open them just like before. No download.

In [ ]:
results = earthaccess.search_data(
    short_name="MUR-JPL-L4-GLOB-v4.1",
    temporal=tspan,
)
print("Daily files found:", len(results))

paths = earthaccess.open(results)

### 6.2 Combine the days and cut out Florida

MUR is a **global 1 km map**, so the files are big, but we only read the Florida box. We cut it out **first** and then average the days. The temperature is stored in **Kelvin** (the scientific scale where 0 is absolute zero), so we subtract 273.15 to get **°C**.

> ⏱️ This step reads a week of 1 km data, so it can take **a minute or two**. For a faster run, use a shorter `tspan` in Step 4 (for example, 2 days).
>
> 🧭 Unlike PACE, MUR's latitude runs **south to north**, so the slice is `lat[0], lat[1]`.

In [ ]:
dataset = xr.open_mfdataset(paths, combine="nested", concat_dim="time")

sst = dataset["analysed_sst"]
sst = sst.sel(lat=slice(lat[0], lat[1]), lon=slice(lon[0], lon[1]))   # cut out Florida
sst = sst.mean("time") - 273.15                                        # average the days, Kelvin -> °C
sst = sst.load()                                                       # read the numbers

print("Florida box:", dict(sst.sizes))
print(f"Coolest: {float(sst.min()):.1f} °C   Warmest: {float(sst.max()):.1f} °C   "
      f"Average: {float(sst.mean()):.1f} °C")

### 6.3 Make the map 🗺️

In [ ]:
fig = plt.figure(figsize=(8, 7))
ax = plt.axes(projection=ccrs.PlateCarree())

mesh = ax.pcolormesh(sst.lon, sst.lat, sst, cmap="turbo", vmin=27, vmax=32,
                     transform=ccrs.PlateCarree())

ax.add_feature(cfeature.LAND, facecolor="#e6dcc8", zorder=2)
ax.coastlines(resolution="10m", linewidth=0.6, zorder=3)
gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
gl.top_labels = gl.right_labels = False

ax.set_extent([lon[0], lon[1], lat[0], lat[1]], crs=ccrs.PlateCarree())
plt.colorbar(mesh, ax=ax, shrink=0.75, extend="both", label="Sea surface temperature (°C)")
ax.set_title(f"Sea Surface Temperature, Florida, {tspan[0]} to {tspan[1]}\nGHRSST MUR (1 km, daily files averaged)")

fig.savefig("maps/florida_sst.png", dpi=150, bbox_inches="tight")
plt.show()

**What do you see?**
- Where is the water warmest? Where is it cooler?
- Shallow water, like around the Keys and Florida Bay, heats up faster than deep water.
- Can you spot the Gulf Stream, the warm current flowing north along Florida's east coast?
- Compare with your chlorophyll map. Do warm and green areas line up, or oppose each other?

> ✏️ **Try it:** change `cmap="turbo"` to `"plasma"` or `"coolwarm"`, or change `vmin` and `vmax`, then re-run the cell.

[back to top](#top)

## 7. Save your maps

Your two maps were saved as picture files in the `maps` folder on the temporary Colab computer. Run the next cell to see them listed.

To keep them, open the 📁 **Files panel** on the left, open the `maps` folder, right-click each image, and choose **Download**. Do this before you close the tab, or they disappear when the session ends.

In [ ]:
for name in sorted(os.listdir("maps")):
    print("Saved:", name)

> 💾 **To keep your own copy of this notebook:** go to **File → Save a copy in Drive**.

[back to top](#top)

## 8. If something goes wrong

| What you see | What to do |
|---|---|
| `NameError: name 'xyz' is not defined` | You skipped a cell. Run the earlier cells **in order** (or **Runtime → Run all**). |
| `ModuleNotFoundError: No module named 'earthaccess'` | The install cell in Step 1 didn't run, or the session reset. Run it again. |
| `Logged in? False` | Wrong username or password. Re-run Step 2. |
| `TypeError: 'dict' object is not callable` | An older example used `item.summary()` with parentheses. In current `earthaccess` it's written without them: `item.summary`. |
| `Files found: 0` (PACE) | Check `tspan` in Step 4. PACE data start in March 2024, and the newest days may not be processed yet. Try a week from a few weeks ago. NASA also renames datasets sometimes: re-run the list in Step 3 and check the short name. |
| `HTTP 401` or `403` error | Your login expired. Re-run Step 2. |
| Opening or reading is slow, or times out | Re-run the cell. NASA's servers can be busy. A shorter `tspan` helps. |
| Colab says "Disconnected" | Reconnect, then **Runtime → Run all**. |
| Map looks empty or all one color | Check your lat/lon box in Step 4. Longitudes west of Greenwich are **negative**. |

Still stuck? Copy the **last line** of the red error message and ask your instructor.

[back to top](#top)

---

## Where to go next

- Change the **time span** to another week or month.
- Change the **box** to the Florida Keys, or to Hawaiʻi `lat=(18.5, 22.5)`, `lon=(-160.6, -154.6)`, or Puerto Rico `lat=(17.6, 18.8)`, `lon=(-67.6, -64.4)`.
- Mark a place you care about by adding `ax.plot(-81.4, 24.55, "w*", ms=14, transform=ccrs.PlateCarree())` (Looe Key reef) to a map cell.

**Data credits:** JPL MUR MEaSUREs Project (2015), GHRSST Level 4 MUR Global Foundation SST Analysis v4.1, PO.DAAC. NASA Ocean Biology Processing Group, PACE OCI Level-3 Biogeochemical (BGC), OB.DAAC. Tutorial format adapted from the NASA OB.DAAC "Orientation to Earthdata Cloud Access" tutorial (A. Windle).